# Visualização da união OR nas máscaras duplicadas

**Objetivo:** documentar o agrupamento de fotografias idênticas e a construção da máscara canônica por união lógica, com inspeção visual das divergências entre referências.

Este notebook abre somente grupos de **treino e validação**. O manifesto e os arquivos do teste congelado não são acessados.

## 1. Agrupamento de imagens e referências

Algumas fotografias aparecem duas vezes com nomes diferentes. Embora a imagem seja idêntica, as máscaras desenhadas pelos especialistas podem variar nas bordas. Manter as duas como amostras independentes daria peso duplo à mesma fotografia e poderia causar vazamento entre partições.

Agrupamos a fotografia uma única vez. Para obter a referência do grupo, a OR considera ferida todo pixel marcado por **A ou B**.

## 2. Localizar o projeto no Google Drive

In [ ]:
from pathlib import Path
import os
from google.colab import drive

drive.mount("/content/drive")
candidates = [
    Path("/content/drive/MyDrive/Projeto_TCC_Lucas"),
    Path("/content/drive/MyDrive/TCC/Projeto_TCC_Lucas"),
]
PROJECT_ROOT = next((p for p in candidates if p.is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(candidates)
os.chdir(PROJECT_ROOT)
print("Projeto:", PROJECT_ROOT)

## 3. Exemplo pequeno da operação OR

O exemplo abaixo mostra diretamente que a interseção guarda apenas o consenso, enquanto a união guarda todas as marcações.

In [ ]:
import numpy as np

mask_a = np.array([[1, 1, 0], [0, 0, 0]], dtype=bool)
mask_b = np.array([[0, 1, 1], [0, 0, 0]], dtype=bool)
print("Máscara A:\n", mask_a.astype(int))
print("Máscara B:\n", mask_b.astype(int))
print("Interseção A AND B:\n", (mask_a & mask_b).astype(int))
print("União A OR B:\n", (mask_a | mask_b).astype(int))

## 4. Conferir a fórmula usada pelo projeto

A função abaixo calcula área, concordância e quanto a OR acrescenta em relação à maior máscara individual.

In [ ]:
import sys, inspect

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from tcc_segmentation.data.duplicate_union import analyze_mask_pair

print(inspect.getsource(analyze_mask_pair))
display(analyze_mask_pair(mask_a, mask_b))

## 5. Quantificar as duplicatas permitidas

O script lê apenas `dfuc_train.csv` e `dfuc_validation.csv`. Os cinco grupos atribuídos ao teste permanecem fechados.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "scripts/analyze_duplicate_or_union.py", "--project-root", str(PROJECT_ROOT)],
    check=True,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)

## 6. Tabela por grupo e resumo

In [ ]:
import json, pandas as pd

OUTPUT = PROJECT_ROOT / "results/data_preparation/or_union_analysis_train_validation"
summary = json.loads((OUTPUT / "or_union_summary.json").read_text(encoding="utf-8"))
per_group = pd.read_csv(OUTPUT / "or_union_per_group.csv")
display(summary)
display(
    per_group[
        [
            "canonical_id",
            "split",
            "dice_between_references",
            "or_added_over_larger_fraction",
            "or_added_over_mean_fraction",
        ]
    ]
    .sort_values("dice_between_references")
    .head(10)
)

## 7. Visualizar três situações

As linhas abaixo representam a menor concordância, um caso intermediário e a maior concordância entre os 34 grupos permitidos.

- vermelho: máscara A;
- azul: máscara B;
- verde: interseção;
- magenta/ciano: discordância;
- amarelo: OR final.

In [ ]:
from PIL import Image

display(Image.open(OUTPUT / "or_union_examples.png"))

## 8. Interpretação do painel

A OR não cria uma nova ferida fora das duas referências: ela reúne os pixels marcados por pelo menos uma delas. Isso reduz omissões entre anotadores, mas adota a borda mais inclusiva. A interseção faria o oposto e poderia reduzir demais a lesão. Escolher apenas A ou B seria arbitrário; usar uma máscara probabilística preservaria incerteza, mas exigiria mudar loss, métricas e protocolo.

A coluna de discordância destaca os pixels em que as referências divergem. O agrupamento usa o conteúdo da fotografia, enquanto a máscara canônica é obtida pela união dos rótulos de suas referências.